<a href="https://colab.research.google.com/github/arturdomitti/pipeline-desambiguacao-ic/blob/main/pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [36]:
!pip install -q networkx requests pandas openai
!pip install -q --upgrade gdown

!gdown 1gKj0uDG5cjRtTlfoClCLrpLJxmmdye5w -O br-capes-colsucup-prod-autor-2021a2024-2025-12-01-bibliografica-artpe-2024.csv
!gdown 1PE2M5lf5ZlMYHVX9DiVggcdzHgocLUfl -O br-capes-colsucup-producao-2021a2024-2025-12-01-bibliografica-artpe-p1.csv
!gdown 1TbIgbheX4Y7b1YIgL_CR7KR0Cf22hWkx -O br-capes-colsucup-producao-2021a2024-2025-12-01-bibliografica-artpe-p2.csv

Using cookies from /root/.cache/gdown/cookies.txt
Downloading...
From (original): https://drive.google.com/uc?id=1gKj0uDG5cjRtTlfoClCLrpLJxmmdye5w
From (redirected): https://drive.google.com/uc?id=1gKj0uDG5cjRtTlfoClCLrpLJxmmdye5w&confirm=t&uuid=4d16126d-8aca-4481-a12a-1dd8ddeff4f8
To: /content/br-capes-colsucup-prod-autor-2021a2024-2025-12-01-bibliografica-artpe-2024.csv
100% 265M/265M [00:01<00:00, 209MB/s]
Using cookies from /root/.cache/gdown/cookies.txt
Downloading...
From (original): https://drive.google.com/uc?id=1PE2M5lf5ZlMYHVX9DiVggcdzHgocLUfl
From (redirected): https://drive.google.com/uc?id=1PE2M5lf5ZlMYHVX9DiVggcdzHgocLUfl&confirm=t&uuid=fabc4c81-5f1d-4331-8184-1fde4dc67d6a
To: /content/br-capes-colsucup-producao-2021a2024-2025-12-01-bibliografica-artpe-p1.csv
100% 326M/326M [00:05<00:00, 58.9MB/s]
Using cookies from /root/.cache/gdown/cookies.txt
Downloading...
From (original): https://drive.google.com/uc?id=1TbIgbheX4Y7b1YIgL_CR7KR0Cf22hWkx
From (redirected): https://dri

In [37]:
import os
import json
import requests
import difflib
import pandas as pd
import networkx as nx
from pprint import pprint
from openai import OpenAI
from getpass import getpass

# Calcula a similaridade lexical entre o nome do docente validado (Sucupira) e do candidato (OpenAlex)
def calculate_name_similarity(name1: str, name2: str) -> float:
    matcher = difflib.SequenceMatcher(None, name1.lower(), name2.lower())
    return float(matcher.ratio())

# Coleta até 5 publicações mais recentes de um determinado autor no OpenAlex, incluindo apenas título, ano e coautores
def fetch_author_works_from_openalex(author_id: str, max_works: int = 5):
    url = "https://api.openalex.org/works"
    params = {
        "filter": f"author.id:{author_id}",
        "per_page": max_works,
        "sort": "publication_year:desc"
    }
    headers = {"User-Agent": "mailto:arturdomitti@usp.br"}

    try:
        response = requests.get(url, params=params, headers=headers, timeout=15)
        if response.status_code != 200:
            return []

        works_data = []
        for work in response.json().get("results", []):
            coauthors = [
                m.get("author", {}).get("display_name", "")
                for m in work.get("authorships", [])
            ]

            works_data.append({
                "title": work.get("title", "Sem título"),
                "year": work.get("publication_year", 2024),
                "coauthors": coauthors
            })
        return works_data
    except Exception:
        return []

# Consulta o OpenAlex em busca de autores candidatos, filtrando apenas aqueles com similaridade lexical >= tau_lex
def fetch_openalex_candidates(target_name: str, tau_lex: float = 0.65, top_n: int = 5):
    print(f"\nConsultando API do OpenAlex para: '{target_name}'...")
    url = "https://api.openalex.org/authors"
    params = {"search": target_name, "per_page": top_n}
    headers = {"User-Agent": "mailto:arturdomitti@usp.br"}
    candidates = []

    try:
        response = requests.get(url, params=params, headers=headers, timeout=15)
        if response.status_code != 200:
            print(f"Erro na API OpenAlex: Status {response.status_code}")
            return []

        data = response.json()
        if not isinstance(data, dict):
            return []

        raw_results = data.get("results")
        if not raw_results or not isinstance(raw_results, list):
            print("Nenhum resultado retornado pelo OpenAlex.")
            return []

        for auth in raw_results:
            if not isinstance(auth, dict):
                continue

            display_name = auth.get("display_name") or ""
            if not display_name:
                continue

            sim = calculate_name_similarity(target_name, display_name)
            if sim < tau_lex:
                continue

            raw_id = auth.get("id") or ""
            auth_id = raw_id.split("/")[-1] if raw_id else ""

            raw_insts = auth.get("last_known_institutions") or []
            affiliations = []
            if isinstance(raw_insts, list):
                for inst in raw_insts:
                    if isinstance(inst, dict) and inst.get("display_name"):
                        affiliations.append(inst.get("display_name"))

            works = fetch_author_works_from_openalex(auth_id) if auth_id else []

            candidates.append({
                "id": auth_id,
                "name": display_name,
                "source": "OpenAlex",
                "validated": False,
                "lexical_similarity": sim,
                "works": works,
                "affiliations": affiliations
            })

        return candidates
    except Exception as e:
        print(f"Erro ao processar dados do OpenAlex: {e}")
        return []

# Carrega e cruza as bases da Sucupira mapeando as colunas exatamente como na tabela original
def load_sucupira_validated_dataset(autores_csv: str, producoes1_csv: str, producoes2_csv: str, ppg_filter: str = "COMPUTAÇÃO", city_filter: str = "SÃO CARLOS"):
    try:
        print("--- CARREGANDO TABELA DE AUTORES DA PRODUÇÃO ---")
        df_autores = pd.read_csv(
            autores_csv,
            sep=";",
            encoding="iso-8859-1",
            low_memory=False,
            on_bad_lines='skip'
        )

        if "NM_PROGRAMA_IES" in df_autores.columns and ppg_filter:
            df_autores = df_autores[df_autores["NM_PROGRAMA_IES"].str.contains(ppg_filter, case=False, na=False)]

        if "NM_MUNICIPIO_PROGRAMA_IES" in df_autores.columns and city_filter:
            df_autores = df_autores[df_autores["NM_MUNICIPIO_PROGRAMA_IES"].str.contains(city_filter, case=False, na=False)]
        elif "SG_ENTIDADE_ENSINO" in df_autores.columns:
            df_autores = df_autores[df_autores["SG_ENTIDADE_ENSINO"].isin(["USP", "UFSCAR"])]

        print(f"Registros de autores filtrados em São Carlos para '{ppg_filter}': {len(df_autores)}")

        print("\n--- CARREGANDO TABELA DE PRODUÇÕES (2 partes) ---")
        df_producoes_p1 = pd.read_csv(
            producoes1_csv,
            sep=";",
            encoding="iso-8859-1",
            low_memory=False,
            on_bad_lines='skip'
        )
        df_producoes_p2 = pd.read_csv(
            producoes2_csv,
            sep=";",
            encoding="iso-8859-1",
            low_memory=False,
            on_bad_lines='skip'
        )

        if set(df_producoes_p1.columns) != set(df_producoes_p2.columns):
            print("[Aviso] As colunas da parte 1 e parte 2 não coincidem!")
            print(f"Só na parte 1: {set(df_producoes_p1.columns) - set(df_producoes_p2.columns)}")
            print(f"Só na parte 2: {set(df_producoes_p2.columns) - set(df_producoes_p1.columns)}")

        df_producoes = pd.concat([df_producoes_p1, df_producoes_p2], ignore_index=True)
        print(f"Total de linhas de produções (p1+p2): {len(df_producoes)}")

        # Mapeamento estrito das chaves de ligação de produção
        col_key_autores = "ID_ADD_PRODUCAO_INTELECTUAL" if "ID_ADD_PRODUCAO_INTELECTUAL" in df_autores.columns else "ID_PRODUCAO_INTELECTUAL"
        col_key_producoes = "ID_ADD_PRODUCAO_INTELECTUAL" if "ID_ADD_PRODUCAO_INTELECTUAL" in df_producoes.columns else "ID_PRODUCAO_INTELECTUAL"

        if col_key_autores not in df_autores.columns or col_key_producoes not in df_producoes.columns:
            print(f"\n[Erro de Estrutura] Colunas disponíveis em Autores: {list(df_autores.columns[:5])}")
            print(f"[Erro de Estrutura] Colunas disponíveis em Produções: {list(df_producoes.columns[:5])}")
            return None

        # Identificação da coluna contendo o título
        colunas_possiveis_titulo = ["NM_PRODUCAO", "DS_TITULO", "NM_TITULO", "NM_PRODUCAO_INTELECTUAL"]
        coluna_titulo = None
        for col in colunas_possiveis_titulo:
            if col in df_producoes.columns:
                coluna_titulo = col
                break

        if not coluna_titulo:
            print(f"[Aviso] Nenhuma coluna de título padrão encontrada. Colunas disponíveis: {list(df_producoes.columns)}")
            coluna_titulo = df_producoes.columns[0]

        # Normaliza valores das chaves para string para prevenir falhas de tipo durante o merge
        df_autores[col_key_autores] = df_autores[col_key_autores].astype(str).str.strip().str.replace(".0", "", regex=False)
        df_producoes[col_key_producoes] = df_producoes[col_key_producoes].astype(str).str.strip().str.replace(".0", "", regex=False)

        print(f"Realizando MERGE entre Autores ({col_key_autores}) e Produções ({col_key_producoes})...")
        df_producoes_subset = df_producoes[[col_key_producoes, coluna_titulo]].drop_duplicates()

        df_merged = pd.merge(
            df_autores,
            df_producoes_subset,
            left_on=col_key_autores,
            right_on=col_key_producoes,
            how="inner"
        )

        print(f"Total de trabalhos validados vinculados aos docentes: {len(df_merged)}")
        return df_merged

    except Exception as e:
        print(f"Erro ao carregar/cruzar bases da Sucupira: {e}")
        return None

# Extrai os trabalhos validados do docente
def fetch_validated_docent_works(df_merged, docente_name: str, max_works: int = 5):
    coluna_autor = "NM_AUTOR" if "NM_AUTOR" in df_merged.columns else "NM_DOCENTE"

    colunas_possiveis_titulo = ["NM_PRODUCAO", "DS_TITULO", "NM_TITULO", "NM_PRODUCAO_INTELECTUAL"]
    coluna_titulo = next((c for c in colunas_possiveis_titulo if c in df_merged.columns), None)
    if coluna_titulo is None:
        raise KeyError(f"Nenhuma coluna de título encontrada em df_merged. Colunas disponíveis: {list(df_merged.columns)}")

    df_docente = df_merged[df_merged[coluna_autor].str.upper() == docente_name.upper()]
    titulos = df_docente[coluna_titulo].dropna().unique()[:max_works]

    works = []
    for t in titulos:
        works.append({
            "title": str(t),
            "year": 2024,
            "source": "Sucupira",
            "validated": True
        })
    return works

# Identifica o tipo de relação existente entre dois nós no grafo
def relation_between(G, u, v):
    relations = []
    if G.has_edge(u, v):
        for _, attrs in G[u][v].items():
            relations.append(attrs["relation"])
    if G.has_edge(v, u):
        for _, attrs in G[v][u].items():
            relations.append(attrs["relation"])
    return sorted(set(relations))

# Converte caminhada do grafo em texto estruturado
def path_to_evidence(G, path):
    parts = []
    for i, node_id in enumerate(path):
        node = G.nodes[node_id]

        node_text = (
            f"{node_id} "
            f"[{node['node_type']}] "
            f"\"{node['label']}\" "
            f"source={node['source']} "
            f"validated={node['validated']}"
        )
        parts.append(node_text)

        if i < len(path) - 1:
            next_node = path[i+1]
            relations = relation_between(G, node_id, next_node)
            parts.append(f"relation={','.join(relations)}")

    return " | ".join(parts)

# Percorre as arestas de saída de um autor e retorna produções associadas
def works_of_author(G, author_id):
    works = []
    for _, target, edge_data in G.out_edges(author_id, data=True):
        if edge_data.get("relation") == "author_of":
            node = G.nodes[target]
            works.append({
                "title": node.get("label"),
                "year": node.get("year"),
                "source": node.get("source"),
                "validated": node.get("validated")
            })
    return works

# Constrói o Grafo de Conhecimento dinâmico
def build_dynamic_graph(validated_docent_name: str, validated_works: list, candidate: dict):
    G = nx.MultiDiGraph()

    def add_node(node_id, node_type, label, source, validated, **attrs):
        G.add_node(node_id, node_type=node_type, label=label, source=source, validated=validated, **attrs)

    def add_edge(source_node, target_node, relation, source, validated, **attrs):
        G.add_edge(source_node, target_node, relation=relation, source=source, validated=validated, **attrs)

    # Entidade e informações do Docente Validado
    add_node("A1", "author", validated_docent_name, "Sucupira", True)
    add_node("P1", "graduate_program", "Programa de Pós-Graduação em Computação", "Sucupira", True)
    add_node("I1", "institution", "Universidade Federal de São Carlos", "Sucupira", True)

    add_edge("A1", "P1", "member_of_ppg", "Sucupira", True)
    add_edge("P1", "I1", "hosted_by", "Sucupira", True)

    # Adiciona as produções validadas do docente no grafo
    for i, work in enumerate(validated_works, start=1):
        work_id = f"W1_{i}"
        add_node(work_id, "work", work["title"], "Sucupira", True, year=work["year"])
        add_edge("A1", work_id, "author_of", "Sucupira", True)

    # Entidade e informações do Candidato
    cand_id = "A2"
    add_node(cand_id, "author", candidate["name"], "OpenAlex", False)

    for aff in candidate.get("affiliations", []):
        if "São Carlos" in aff or "UFSCar" in aff or "USP" in aff:
            add_edge(cand_id, "I1", "affiliated_with", "OpenAlex", False)
        else:
            inst_id = f"I_{abs(hash(aff)) % 10000}"
            add_node(inst_id, "institution", aff, "OpenAlex", False)
            add_edge(cand_id, inst_id, "affiliated_with", "OpenAlex", False)

    for i, work in enumerate(candidate.get("works", []), start=1):
        work_id = f"W2_{i}"
        add_node(work_id, "work", work["title"], "OpenAlex", False, year=work["year"])
        add_edge(cand_id, work_id, "author_of", "OpenAlex", False)

        for coauthor in work.get("coauthors", []):
            if coauthor != candidate["name"]:
                co_id = f"A_co_{abs(hash(coauthor)) % 10000}"
                if not G.has_node(co_id):
                    add_node(co_id, "author", coauthor, "OpenAlex", False)
                add_edge(co_id, work_id, "author_of", "OpenAlex", False)

    return G, "A1", cand_id

# Fluxo de execução principal
if __name__ == "__main__":
    # Nomes dos arquivos de Autores e de Produções
    autores_csv = "br-capes-colsucup-prod-autor-2021a2024-2025-12-01-bibliografica-artpe-2024.csv"
    producoes1_csv = "br-capes-colsucup-producao-2021a2024-2025-12-01-bibliografica-artpe-p1.csv"
    producoes2_csv = "br-capes-colsucup-producao-2021a2024-2025-12-01-bibliografica-artpe-p2.csv"

    print("--- INICIANDO PIPELINE DE DESAMBIGUAÇÃO SUCUPIRA-OPENALEX ---")
    df_merged = load_sucupira_validated_dataset(
        autores_csv=autores_csv,
        producoes1_csv=producoes1_csv,
        producoes2_csv=producoes2_csv,
        ppg_filter="COMPUTAÇÃO",
        city_filter="SÃO CARLOS"
    )

    if df_merged is not None and not df_merged.empty:
        coluna_autor = "NM_AUTOR" if "NM_AUTOR" in df_merged.columns else "NM_DOCENTE"
        docentes_amostra = df_merged[coluna_autor].dropna().unique()[:5]

        print(f"\nAmostra de docentes selecionados para desambiguação: {list(docentes_amostra)}\n")

        BASE_URL = "https://agents4gov.icmc.usp.br/api/v1"
        MODEL = "externo.revejo-qwen3.6-35b-a3b"

        API_KEY = getpass("Insira a sua API Key do Agents4Gov: ").strip()

        if not API_KEY:
            raise ValueError("API Key não pode ser vazia!")

        client = OpenAI(api_key=API_KEY, base_url=BASE_URL)

        SYSTEM_PROMPT = """
        Você realiza resolução de entidades em grafos de conhecimento.

        A primeira entidade é um pesquisador pertencente ao conjunto validado.
        A segunda entidade é um candidato ainda não validado.

        Analise as evidências fornecidas.

        Primeiro infira os principais tópicos de pesquisa de cada autor usando exclusivamente os títulos das produções.

        Depois analise conjuntamente nome, produções, coautores, instituições, vínculos e caminhos encontrados no grafo.

        Similaridade temática é uma evidência adicional.
        Ela não deve ser utilizada isoladamente para confirmar ou rejeitar uma identidade.

        Retorne exclusivamente um objeto JSON válido.

        O objeto deve conter os campos:
        same_person, confidence, validated_author_topics, candidate_author_topics, topic_evidence, structural_evidence, contrary_evidence, reasoning_summary, decision

        decision deve assumir um dos valores: validate_candidate, reject_candidate, uncertain
        """.strip()

        for docente in docentes_amostra:
            print("=" * 70)
            print(f"PROCESSANDO DOCENTE VALIDADO: {docente}")
            print("=" * 70)

            validated_works = fetch_validated_docent_works(df_merged, docente)

            candidatos = fetch_openalex_candidates(docente, tau_lex=0.65)
            print(f"Total de candidatos encontrados no OpenAlex: {len(candidatos)}\n")

            for c in candidatos:
                print(f"--> Analisando Candidato OpenAlex: {c['name']} (ID: {c['id']})")

                G, source_id, target_id = build_dynamic_graph(docente, validated_works, c)

                UG = nx.Graph(G)
                MAX_DEPTH = 5
                paths = list(nx.all_simple_paths(UG, source=source_id, target=target_id, cutoff=MAX_DEPTH))

                path_evidence = [path_to_evidence(G, p) for p in paths]
                validated_author_works = works_of_author(G, source_id)
                candidate_author_works = works_of_author(G, target_id)

                context = {
                    "validated_author": {
                        "name": docente,
                        "source": "Sucupira",
                        "works": validated_author_works
                    },
                    "candidate_author": {
                        "name": c["name"],
                        "source": "OpenAlex",
                        "works": candidate_author_works
                    },
                    "graph_paths": path_evidence,
                    "known_validated_facts": [
                        {"subject": source_id, "relation": "member_of_ppg", "object": "P1"},
                        {"subject": "P1", "relation": "hosted_by", "object": "I1"}
                    ]
                }

                USER_PROMPT = f"Analise o seguinte par de pesquisadores:\n\nCONTEXTO:\n{json.dumps(context, ensure_ascii=False, indent=2)}".strip()
                print(USER_PROMPT)

                print("Enviando evidências para a LLM...")
                try:
                    response = client.chat.completions.create(
                        model=MODEL,
                        messages=[
                            {"role": "system", "content": SYSTEM_PROMPT},
                            {"role": "user", "content": USER_PROMPT}
                        ],
                        temperature=0
                    )
                    content = response.choices[0].message.content
                    result = json.loads(content)
                    print("\n--- DECISÃO DA LLM ---")
                    pprint(result)
                    print("-" * 50)
                except Exception as e:
                    print(f"Erro ao consultar a LLM: {e}\n")
    else:
        print("\n[Aviso] Não foi possível carregar a lista de docentes e produções. Verifique os caminhos dos arquivos CSV.")

--- INICIANDO PIPELINE DE DESAMBIGUAÇÃO SUCUPIRA-OPENALEX ---
--- CARREGANDO TABELA DE AUTORES DA PRODUÇÃO ---
Registros de autores filtrados em São Carlos para 'COMPUTAÇÃO': 438

--- CARREGANDO TABELA DE PRODUÇÕES (2 partes) ---
Total de linhas de produções (p1+p2): 1290664
Realizando MERGE entre Autores (ID_ADD_PRODUCAO_INTELECTUAL) e Produções (ID_ADD_PRODUCAO_INTELECTUAL)...
Total de trabalhos validados vinculados aos docentes: 438

Amostra de docentes selecionados para desambiguação: ['JOAO AGOSTINHO MACHADO NETO', 'TIAGO AGOSTINHO DE ALMEIDA', 'FABIANO CUTIGI FERRARI', 'CAROLINE APARECIDA DE PAULA SILVA', 'ORLANDO LEE']

Insira a sua API Key do Agents4Gov: ··········
PROCESSANDO DOCENTE VALIDADO: JOAO AGOSTINHO MACHADO NETO

Consultando API do OpenAlex para: 'JOAO AGOSTINHO MACHADO NETO'...
Total de candidatos encontrados no OpenAlex: 5

--> Analisando Candidato OpenAlex: João Agostinho Machado‐Neto (ID: A5004866942)
Analise o seguinte par de pesquisadores:

CONTEXTO:
{
  "valida